In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

c:\Users\Manoj\Python ML\Lib\site-packages\torch\cuda\__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


****Standard MoE Layer****

The Standard MoE approach follows the design pattern established by GShard and Switch Transformers:

**Router**: A linear layer that outputs routing probabilities for each expert


**Top-K Selection**: Each token is routed to the k highest-probability experts

**Expert Processing**: Selected experts process the tokens

**Weighted Combination**: Expert outputs are combined according to router probabilities

**Auxiliary Load Balancing Loss**: Encourages uniform expert utilization

In [4]:
d_model = 512

# MoE Specific Params for Fine-Grained Expert Segmentation
n_routed_experts: int = 16
top_k: int = 2
expert_hidden_dim: int = 512 # (n_embd * 4) / 4
dropout = 0.0

In [5]:
class Expert(nn.Module):
    def __init__(self, n_embd: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, hidden_dim, bias=False),
            nn.GELU(),
            nn.Linear(hidden_dim, n_embd, bias=False),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

In [6]:
class StandardMOELayer(nn.Module):
  def __init__(self, n_routed_experts, top_k, expert_hidden_dim, dropout=0.0):
    super().__init__()
    self.n_routed_experts = n_routed_experts
    self.top_k = top_k
    # Module List of experts
    self.experts = nn.ModuleList(
        [Expert(d_model, expert_hidden_dim, dropout=dropout) for _ in range(self.n_routed_experts)]
      )
    # router matrix (d_model, no_of_experts)
    self.router_matrix = nn.Linear(d_model, self.n_routed_experts)
    self.aux_loss_coeff = 1e-2
  
  def forward(self, x):
    B, T, C = x.shape  #(batch_size, seq_len, d_model)

    # Reshape the input to [N, C], where N = B * S (number of tokens).
    x_flat = x.view(-1, C) 

    num_tokens = x_flat.shape[0]

    router_logits = self.router_matrix(x_flat)  # (n_tokens, col=no.of_Experts) each column belongs to each expert
    routing_weights = self.F.softmax(router_logits, dim=-1, dtype=torch.float)  
    topk_weights, topk_indices = torch.topk(routing_weights, self.top_k, dim=-1)  # we are selecting topk experts for each token
    gates = topk_weights / topk_weights.sum(dim=-1, keepdim=True)   

    # --- CORRECTED Auxiliary Load Balancing Loss (as per the book) ---
    # f_i: fraction of tokens dispatched to expert i
    # p_i: fraction of router probability allocated to expert i

    # calculate the f_i for each expert
    expert_counts = torch.zeros(self.n_routed_experts, device=x.device)
    expert_counts.index_add_(0, topk_indices.view(-1), torch.ones_like(topk_indices.view(-1), dtype=torch.float))
    f_i = expert_counts / num_tokens

    # calculate = p_i for each expert 
    p_i = routing_weights.mean(dim=0)
    # print(p_i)

    # The aux_loss = is the dot product of these two distributions, scaled
    aux_loss = self.n_routed_experts * self.aux_loss_coeff * torch.sum(p_i * f_i) 

    # Final output
    # Dispatch tokens
    final_output_flat = torch.zeros_like(x_flat)
    for i in range(self.n_routed_experts):
      mask = (topk_indices == i)
      row_idx, which_k = mask.nonzero(as_tuple=True)
      if row_idx.numel() == 0: continue
      
      expert_in = x_flat.index_select(0, row_idx)
      expert_out = self.experts[i](expert_in)
      gate_values = gates[row_idx, which_k].unsqueeze(1)
            
      final_output_flat.index_add_(0, row_idx, expert_out * gate_values)
            
    return final_output_flat.view(B, T, C), aux_loss      # aux_loss will be added to the main training loss

In [7]:
class DummyTransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm()
        self.attn = MultiHeadAttention()
        self.ln_2 = nn.LayerNorm()
        self.moe = StandardMoELayer(n_routed_experts, top_k, expert_hidden_dim, dropout=0.0)   # in trf blocks instead FFNN now MoE

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        moe_out, aux_loss = self.moe(self.ln_2(x))
        x = x + moe_out
        return x, aux_loss